# Hybrid retrieval
Do sparse and dense rankings recover different papers?

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
smoke = os.environ.get('PYTHONIC_SMOKE') == '1'

In [ ]:
from pythonic.data import fixtures, load_papers, split_papers
from pythonic.dense import BiomedicalEmbedder, DenseIndex, HybridIndex
from pythonic.retrieval import BM25, retrieval_metrics
import numpy as np
if smoke:
    papers = fixtures()
    def embed(texts):
        return np.array([[text.lower().count(word) for word in ('fever', 'fatigue', 'recovery')] for text in texts], dtype=np.float32).reshape(-1, 3)
    questions = papers
else:
    os.environ.setdefault('HF_HOME', str(root / 'data/hf'))
    papers = load_papers(root / 'data/pubmedqa.json')
    embed = BiomedicalEmbedder.load('cuda' if torch.cuda.is_available() else 'cpu')
    questions = split_papers(papers).test
sections = [section for paper in papers for section in paper.sections]
sparse = BM25(sections)
dense = DenseIndex(sections, embed)
for name, index in [('sparse', sparse), ('dense', dense), ('hybrid', HybridIndex(sparse, dense))]:
    print(name, retrieval_metrics(index, questions))

Offline smoke mode uses three explicit lexical features. The real run uses mean-pooled frozen BiomedBERT; it is a baseline, not a retrieval-tuned encoder.